downloading plant files of zip and then coping it to new folder plant_dataset

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
import os

# 1. Download only the data.zip file directly to /content
print("🚀 Downloading data.zip from Hugging Face (this might take a minute)...")
zip_path = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="data.zip",
    repo_type="dataset",
    local_dir="/content"
)

# 2. Extract the downloaded zip file
extract_path = "/content/plant_village_data"
print(f"📦 Extracting files to {extract_path}...")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Done! Your dataset is fully extracted and ready to use.")

🚀 Downloading data.zip from Hugging Face (this might take a minute)...


data.zip:   0%|          | 0.00/2.18G [00:00<?, ?B/s]

📦 Extracting files to /content/plant_village_data...
✅ Done! Your dataset is fully extracted and ready to use.


In [ ]:
import os
import shutil

# Define source and destination paths
source_dir = "/content/plant_village_data/raw/color"
destination_dir = "/content/plant_dataset"

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Get all the class folders (Apple___Apple_scab, Blueberry___healthy, etc.)
class_folders = [f for f in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, f))]

print(f"🔄 Found {len(class_folders)} classes to copy. Starting transfer...\n")

for folder_name in class_folders:
    src_path = os.path.join(source_dir, folder_name)
    dest_path = os.path.join(destination_dir, folder_name)

    # Check if the folder already exists in the destination to avoid errors
    if not os.path.exists(dest_path):
        print(f"📁 Copying: {folder_name} -> {destination_dir}")
        shutil.copytree(src_path, dest_path)
    else:
        print(f"⚠️ {folder_name} already exists in destination, skipping.")

print(f"\n✅ All set! All class folders are now neatly organized under '{destination_dir}'.")

🔄 Found 38 classes to copy. Starting transfer...

📁 Copying: Tomato___Septoria_leaf_spot -> /content/plant_dataset
📁 Copying: Cherry_(including_sour)___healthy -> /content/plant_dataset
📁 Copying: Blueberry___healthy -> /content/plant_dataset
📁 Copying: Tomato___Bacterial_spot -> /content/plant_dataset
📁 Copying: Tomato___Tomato_mosaic_virus -> /content/plant_dataset
📁 Copying: Tomato___Tomato_Yellow_Leaf_Curl_Virus -> /content/plant_dataset
📁 Copying: Strawberry___Leaf_scorch -> /content/plant_dataset
📁 Copying: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot -> /content/plant_dataset
📁 Copying: Peach___healthy -> /content/plant_dataset
📁 Copying: Corn_(maize)___Common_rust_ -> /content/plant_dataset
📁 Copying: Tomato___Leaf_Mold -> /content/plant_dataset
📁 Copying: Grape___Black_rot -> /content/plant_dataset
📁 Copying: Pepper,_bell___healthy -> /content/plant_dataset
📁 Copying: Grape___healthy -> /content/plant_dataset
📁 Copying: Soybean___healthy -> /content/plant_dataset
📁 Copyi

Downloading rice dataset and renaming files for structured dataset

In [ ]:
import kagglehub
import shutil
import os

# Plant Disease
# plant_path = kagglehub.dataset_download("emmarex/plantdisease")

# Rice Disease
rice_path = kagglehub.dataset_download("dedeikhsandwisaputra/rice-leafs-disease-dataset")

# Sugarcane Disease
# sugar_path = kagglehub.dataset_download("akilesh253/sugarcane-plant-diseases-dataset")

# print(plant_path)
print(rice_path)
# print(sugar_path)

# Copy to content for easier access
# shutil.copytree(plant_path, "/content/plant_dataset", dirs_exist_ok=True)
shutil.copytree(rice_path, "/content/rice_dataset", dirs_exist_ok=True)
# shutil.copytree(sugar_path, "/content/sugar_dataset", dirs_exist_ok=True)

Using Colab cache for faster access to the 'rice-leafs-disease-dataset' dataset.
/kaggle/input/rice-leafs-disease-dataset


'/content/rice_dataset'

In [ ]:
!rm -rf /content/combined_dataset/

In [ ]:
import os
import glob
import shutil

# Define paths based on your folder structure
base_dir = "/content/rice_dataset"
train_dir = os.path.join(base_dir, "RiceLeafsDisease/train")
val_dir = os.path.join(base_dir, "RiceLeafsDisease/validation")

# Automatically grabs all 6 class folders inside the directory
classes = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]

print(f"📁 Found classes to process: {classes}\n")

# Loop through EACH class one by one
for class_name in classes:
    class_train_path = os.path.join(train_dir, class_name)
    class_val_path = os.path.join(val_dir, class_name)

    if not os.path.exists(class_val_path):
        print(f"⚠️ Validation folder for {class_name} not found, skipping.")
        continue

    # 1. Count existing files in this specific train folder to find the starting index
    existing_train_files = glob.glob(os.path.join(class_train_path, "*"))
    start_index = len(existing_train_files) + 1

    # 2. Get all images from this specific validation class folder
    val_files = glob.glob(os.path.join(class_val_path, "*"))

    print(f"🔄 Processing [{class_name}]: {len(existing_train_files)} existing files. Merging {len(val_files)} validation files starting at index ({start_index})...")

    # Rename and move each image for the current class
    for val_file in val_files:
        ext = os.path.splitext(val_file)[1]

        # Dynamically matches the file name to its current class
        new_filename = f"{class_name} ({start_index}){ext}"
        dest_path = os.path.join(class_train_path, new_filename)

        shutil.move(val_file, dest_path)
        start_index += 1

print("\n📦 Moving folders directly under /content/rice_dataset...")
# 3. Restructure: Move all 6 processed folders directly under rice_dataset
inner_root = os.path.join(base_dir, "RiceLeafsDisease")
for class_name in classes:
    old_location = os.path.join(train_dir, class_name)
    new_location = os.path.join(base_dir, class_name)
    shutil.move(old_location, new_location)

# 4. Clean up the now empty intermediate directories
shutil.rmtree(inner_root)

print("✅ All done! Each of the 6 classes has been successfully merged and clean-indexed!")

📁 Found classes to process: ['narrow_brown_spot', 'leaf_blast', 'brown_spot', 'leaf_scald', 'bacterial_leaf_blight', 'healthy']

🔄 Processing [narrow_brown_spot]: 350 existing files. Merging 88 validation files starting at index (351)...
🔄 Processing [leaf_blast]: 350 existing files. Merging 88 validation files starting at index (351)...
🔄 Processing [brown_spot]: 350 existing files. Merging 88 validation files starting at index (351)...
🔄 Processing [leaf_scald]: 350 existing files. Merging 88 validation files starting at index (351)...
🔄 Processing [bacterial_leaf_blight]: 350 existing files. Merging 88 validation files starting at index (351)...
🔄 Processing [healthy]: 350 existing files. Merging 88 validation files starting at index (351)...

📦 Moving folders directly under /content/rice_dataset...
✅ All done! Each of the 6 classes has been successfully merged and clean-indexed!


Now check folder structure

In [ ]:
import os

def show_structure(root, max_depth=2):

    for dirpath, dirnames, filenames in os.walk(root):

        depth = dirpath[len(root):].count(os.sep)

        if depth > max_depth:
            continue

        print(dirpath)
        print("Folders:", dirnames[:5])
        print("Files:", filenames[:5])
        print("-"*50)

show_structure("/content/plant_dataset")
show_structure("/content/rice_dataset")
show_structure("/content/sugar_dataset")

/content/plant_dataset
Folders: ['Tomato___Septoria_leaf_spot', 'Cherry_(including_sour)___healthy', 'Blueberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Tomato_mosaic_virus']
Files: []
--------------------------------------------------
/content/plant_dataset/Tomato___Septoria_leaf_spot
Folders: []
Files: ['71d8607e-536c-4384-a9c5-db2dcd3e149d___JR_Sept.L.S 2641.JPG', 'b49c2ffd-6fa4-4ab8-8458-f4cc444c1325___Matt.S_CG 6007.JPG', '6f564ce5-7595-469a-8db2-cfa8523ae647___Matt.S_CG 1546.JPG', '43f78d0c-798d-4466-83f6-49e84d6a9fad___JR_Sept.L.S 2473.JPG', 'f4360fdc-ebcc-4ecd-8873-4fe37484e2dd___Matt.S_CG 0825.JPG']
--------------------------------------------------
/content/plant_dataset/Cherry_(including_sour)___healthy
Folders: []
Files: ['515f8058-9e91-4b9a-b74a-deb114fc3f39___JR_HL 9444.JPG', '5b04b29d-0171-44d6-a118-87e0d17b9391___JR_HL 4234.JPG', 'ecf911e8-82ab-437b-b3a9-61ee8429e3ba___JR_HL 9705.JPG', '17d3b0b7-b847-4510-a2e0-30b9314284f3___JR_HL 9556.JPG', 'f2a6aa00-cab4-4267-

In [ ]:
import os
import shutil

combined_dir = "/content/combined_dataset"

# remove old combined dataset
if os.path.exists(combined_dir):
    shutil.rmtree(combined_dir)

os.makedirs(combined_dir)

# ======================
# PlantVillage
# ======================

plant_root = "/content/plant_dataset"

for cls in os.listdir(plant_root):

    src = os.path.join(plant_root, cls)

    if not os.path.isdir(src):
        continue

    dst = os.path.join(combined_dir, cls)

    shutil.copytree(src, dst)

# ======================
# Rice
# ======================

rice_root = "/content/rice_dataset"

for cls in os.listdir(rice_root):

    src = os.path.join(rice_root, cls)

    if not os.path.isdir(src):
        continue

    new_cls = f"Rice_{cls}"

    dst = os.path.join(combined_dir, new_cls)

    shutil.copytree(src, dst)

# ======================
# Sugarcane
# ======================

sugar_root = "/content/sugar_dataset/Sugarcane_leafs"

for cls in os.listdir(sugar_root):

    src = os.path.join(sugar_root, cls)

    if not os.path.isdir(src):
        continue

    new_cls = f"Sugarcane_{cls}"

    dst = os.path.join(combined_dir, new_cls)

    shutil.copytree(src, dst)

print("Dataset Combined Successfully")

Dataset Combined Successfully


In [ ]:
classes = sorted(os.listdir("/content/combined_dataset"))

print("Total Classes =", len(classes))

for cls in classes:
    print(cls)

Total Classes = 50
Apple___Apple_scab
Apple___Black_rot
Apple___Cedar_apple_rust
Apple___healthy
Blueberry___healthy
Cherry_(including_sour)___Powdery_mildew
Cherry_(including_sour)___healthy
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Corn_(maize)___Common_rust_
Corn_(maize)___Northern_Leaf_Blight
Corn_(maize)___healthy
Grape___Black_rot
Grape___Esca_(Black_Measles)
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
Grape___healthy
Orange___Haunglongbing_(Citrus_greening)
Peach___Bacterial_spot
Peach___healthy
Pepper,_bell___Bacterial_spot
Pepper,_bell___healthy
Potato___Early_blight
Potato___Late_blight
Potato___healthy
Raspberry___healthy
Rice_bacterial_leaf_blight
Rice_brown_spot
Rice_healthy
Rice_leaf_blast
Rice_leaf_scald
Rice_narrow_brown_spot
Soybean___healthy
Squash___Powdery_mildew
Strawberry___Leaf_scorch
Strawberry___healthy
Sugarcane_BacterialBlights
Sugarcane_Healthy
Sugarcane_Mosaic
Sugarcane_RedRot
Sugarcane_Rust
Sugarcane_Yellow
Tomato___Bacterial_spot
Tomato___Early_bl

In [ ]:
import pandas as pd
import os

data = []

for cls in sorted(os.listdir("/content/combined_dataset")):

    cnt = len(os.listdir(
        os.path.join(
            "/content/combined_dataset",
            cls
        )
    ))

    data.append([cls,cnt])

df = pd.DataFrame(
    data,
    columns=["Class","Images"]
)

df.sort_values(
    by="Images",
    ascending=False
)

,Class,Images
15,Orange___Haunglongbing_(Citrus_greening),5507
47,Tomato___Tomato_Yellow_Leaf_Curl_Virus,5357
30,Soybean___healthy,5090
34,Sugarcane_BacterialBlights,4800
35,Sugarcane_Healthy,3132
37,Sugarcane_RedRot,3108
38,Sugarcane_Rust,3084
39,Sugarcane_Yellow,3030
36,Sugarcane_Mosaic,2772
16,Peach___Bacterial_spot,2297


#Training pipeline

##Create Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
import os
import shutil

SOURCE = "/content/combined_dataset"
DEST = "/content/final_dataset"

if os.path.exists(DEST):
    shutil.rmtree(DEST)

for split in ["train","val","test"]:
    os.makedirs(os.path.join(DEST,split))

for cls in os.listdir(SOURCE):

    class_dir = os.path.join(SOURCE,cls)

    if not os.path.isdir(class_dir):
        continue

    images = os.listdir(class_dir)

    train_imgs,temp_imgs = train_test_split(
        images,
        test_size=0.30,
        random_state=42
    )

    val_imgs,test_imgs = train_test_split(
        temp_imgs,
        test_size=0.50,
        random_state=42
    )

    splits = {
        "train":train_imgs,
        "val":val_imgs,
        "test":test_imgs
    }

    for split_name,split_imgs in splits.items():

        dst = os.path.join(
            DEST,
            split_name,
            cls
        )

        os.makedirs(dst)

        for img in split_imgs:

            shutil.copy(
                os.path.join(class_dir,img),
                os.path.join(dst,img)
            )

In [ ]:
!du -sh /content/final_dataset/

2.3G	/content/final_dataset/


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torchvision.models import efficientnet_b0

from torch.utils.data import DataLoader

from collections import Counter

import numpy as np

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


##Transformations

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomVerticalFlip(),

    transforms.RandomRotation(20),

    transforms.RandomPerspective(),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3
    ),

    transforms.ToTensor()
])

val_transform = transforms.Compose([

    transforms.Resize((224,224)),
    transforms.ToTensor()

])

In [ ]:
train_dataset = datasets.ImageFolder(
    "/content/final_dataset/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "/content/final_dataset/val",
    transform=val_transform
)

test_dataset = datasets.ImageFolder(
    "/content/final_dataset/test",
    transform=val_transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
targets = train_dataset.targets

counter = Counter(targets)

weights = []

for i in range(len(counter)):

    weights.append(
        len(targets) /
        (len(counter)*counter[i])
    )

weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

In [ ]:
model = efficientnet_b0(
    weights="DEFAULT"
)

num_classes = len(
    train_dataset.classes
)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [ ]:
def evaluate(model,loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images,labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _,pred = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                pred==labels
            ).sum().item()

    return 100*correct/total

In [ ]:
epochs = 15

best_acc = 0

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images,labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    val_acc = evaluate(
        model,
        val_loader
    )

    print(
        f"Epoch {epoch+1}"
    )

    print(
        f"Loss = {running_loss:.4f}"
    )

    print(
        f"Val Acc = {val_acc:.2f}%"
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

print(
    "Best Accuracy:",
    best_acc
)

Epoch 1
Loss = 1181.1421
Val Acc = 95.30%
Epoch 2
Loss = 285.0204
Val Acc = 96.57%
Epoch 3
Loss = 195.1935
Val Acc = 97.13%
Epoch 4
Loss = 160.4887
Val Acc = 97.29%
Epoch 5
Loss = 139.9251
Val Acc = 97.76%
Epoch 6
Loss = 125.5994
Val Acc = 97.79%
Epoch 7
Loss = 109.2175
Val Acc = 98.26%
Epoch 8
Loss = 99.5336
Val Acc = 98.07%
Epoch 9
Loss = 90.6884
Val Acc = 98.08%
Epoch 10
Loss = 84.5681
Val Acc = 98.47%
Epoch 11
Loss = 80.0606
Val Acc = 98.15%
Epoch 12
Loss = 66.3053
Val Acc = 98.19%
Epoch 13
Loss = 72.0750
Val Acc = 98.28%
Epoch 14
Loss = 64.9095
Val Acc = 98.28%
Epoch 15
Loss = 66.3540
Val Acc = 98.59%
Best Accuracy: 98.58629661751951


In [ ]:
model.load_state_dict(
    torch.load(
        "best_model.pth"
    )
)

test_acc = evaluate(
    model,
    test_loader
)

print(
    "Test Accuracy:",
    test_acc
)

Test Accuracy: 98.58862239154905


In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_true = []
y_pred = []

model.eval()

with torch.no_grad():

    for images,labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _,pred = torch.max(
            outputs,
            1
        )

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            pred.cpu().numpy()
        )

In [ ]:
print(

    classification_report(
        y_true,
        y_pred,
        target_names=
        test_dataset.classes
    )

)

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       1.00      1.00      1.00        95
                                 Apple___Black_rot       1.00      0.99      0.99        94
                          Apple___Cedar_apple_rust       1.00      1.00      1.00        42
                                   Apple___healthy       1.00      1.00      1.00       247
                               Blueberry___healthy       1.00      1.00      1.00       226
          Cherry_(including_sour)___Powdery_mildew       1.00      1.00      1.00       158
                 Cherry_(including_sour)___healthy       0.99      0.98      0.99       129
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.96      1.00      0.98        77
                       Corn_(maize)___Common_rust_       1.00      1.00      1.00       179
               Corn_(maize)___Northern_Leaf_Blight       1.00      0.98      0.

###Save Class Names

In [ ]:
import json

with open(
    "classes.json",
    "w"
) as f:

    json.dump(
        train_dataset.classes,
        f
    )